In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import matplotlib.colors as colors
# import holoviews as hv
import gzip
from matplotlib.patches import Rectangle

import matplotlib as mpl
from cycler import cycler
# hv.extension('bokeh')
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.signal import peak_widths
from scipy.signal import savgol_filter
from numpy.fft import rfft, rfftfreq

In [ ]:
from scipy.signal import savgol_filter
import gzip
import matplotlib as mpl
from cycler import cycler
# hv.extension('bokeh')
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.signal import peak_widths
from numpy.fft import rfft, rfftfreq

import skunk

In [ ]:
# Gain table
q1 = 100.075
q2 = 100.103
q3 = 100.087
q4 = 100.082

s1 = 100.104
s2 = 99.977
s3 = 100.183

In [ ]:
def load1dFig5(i):
    return np.loadtxt(f'../20221123 HMIA 13-3 Kondo/data/{i}/data.tsv')


def load2dFig5(i, num):
    tmp = np.loadtxt(f'../20221123 HMIA 13-3 Kondo/data/{i}/data.tsv')
    numrows = np.floor(tmp.shape[0] / num)
    data_dict = {}
    for j in range(tmp.shape[1]):
        data_dict[str(j)] = tmp[:num*int(numrows), j].reshape((int(numrows), num)).transpose()
    return data_dict
# fast axis is column-wise

In [ ]:
def fitG(deltaVg, G0, x):
    return G0*((1e-6+deltaVg*x)/(1e-6+np.sinh(deltaVg*x)))

In [ ]:
# %%
# fig, ax = plt.subplots(2, 3, figsize=(10, 8),gridspec_kw={"width_ratios":[1,1, 0.001],"height_ratios":[1, 1]})
# ax1 = ax[0,0]
# ax2 = ax[0,1]
# ax3 = ax[1,0]
# ax4 = ax[1,1]
# figinset, inset_ax = plt.subplots()
figS = plt.figure(figsize=(16, 8),constrained_layout=True)
gs = figS.add_gridspec(2, 4, width_ratios=(1,1,1,1))
# gs = GridSpec(3, 3, figure=fig1)
fS_ax1 = figS.add_subplot(gs[:1, :2])
# f3_ax1.set_title('gs[0, :3]')
fS_ax2 = figS.add_subplot(gs[:1, 2:4])
fS_ax4 = figS.add_subplot(gs[1:2, 3])
# inset_ax = fig2.add_subplot(gs[:1, 2])
# f1_ax2cbar = fig1.add_subplot(gs[0, 3])
# plt.setp(f4_ax2.get_yticklabels(), visible=False)
# f3_ax2.set_title('gs[0, 3:]')
fS_ax3 = figS.add_subplot(gs[1:2,:3])
fS_ax1.text(-0.1, 1, "(a)", fontsize=14, va="bottom", ha="right", transform=fS_ax1.transAxes)

fS_ax2.text(-0.1, 1, "(b)", fontsize=14, va="bottom", ha="right", transform=fS_ax2.transAxes)
fS_ax1.set_axis_off()
# skunk.connect(f1_ax3, 'zoom')
# f2_ax3.set_axis_off()


# ax2=ax.twinx()
offset=0
npoints=[20]
colors = ['b', 'r' ,'k']
for i, d in enumerate([176]):
    dat = load1dFig5(d)
    vg = dat[:, 1]
    vplunger = vg
#     vs = dat[:, 2]*100/q1
    vt = dat[:, 4]*100/q2
    vr = dat[:, 2]*100/q1

    g = 3*vt/(vt+vr)
    G = g
    alpha = 0.16/5.9
    widths = np.zeros((58))
    width_height = np.zeros((58))
    left = np.zeros((58))
    right = np.zeros((58))
    start=2
    peaks, _ = find_peaks(G[start:], prominence=0.1)
    Gpeak = G[start+peaks]
    Gmax = np.max(G[start:])
    print(np.diff(vplunger[start + peaks]))
#         phase = dat[:, 3]
#         v = 5e-6
#         vqpc = -1.988 - 0.002*i
    widths[:len(peaks)], width_height[:len(peaks)], left[:len(peaks),], right[:len(peaks)] = peak_widths(G[start:], peaks, rel_height=0.5)
#         ax.plot(1*0.03*i+curr[730:930]/v*25813, lw=0.75, label=str(vqpc)+' V', color='k')
    # ax.plot(vplunger[start:], offset*0.005*i+G[start:], 'o-', lw=0.75, markersize=2, label=labels[i])
    
#         ax.plot(1*0.03*i+curr[left[i]]/v*25813, lw=0.75, label=str(vqpc)+' V', color='k')
    # ax.semilogy(vplunger[start+peaks],offset*0.005*i+G[start+peaks],'b*')

    for peaknum in range(0,5):
        Vg0 = vplunger[start+ peaks[peaknum]]
#         print(Vg0)
        G0 = np.max(G[start+peaks[peaknum]])
#         alpha = 0.04
        Vg = vplunger[start + peaks[peaknum] - npoints[i] : start + peaks[peaknum] + npoints[i]]
        deltaVg = Vg - Vg0
        Gfit = G[start + peaks[peaknum] - npoints[i] : start + peaks[peaknum] + npoints[i]]
        popt, pcov = curve_fit(fitG, deltaVg, Gfit, p0=[G0, 11000])
#             popt2, pcov2 = curve_fit(fitGdot, deltaVg, Gfit, p0=[G0, 5000])
        print(alpha*1*(popt[1]**-1)*1e6/86)
#         print(alpha*0.5*(popt2[1]**-1)*1e6/86)

#         print(popt[1])
        # ax.plot(Vg, 0*0.005*i+fitG(deltaVg, popt[0], popt[1]), 'ro-', lw=1)#, label='SET')

    fS_ax2.plot(vplunger[:]+ 0.0*(i//2),0.0*i+ G[:], 'bx-', lw=1, markersize=2)
    # f2_ax2.legend(loc = 'center left')
    # f2_ax2.legend(loc = 'center left')

fS_ax2.grid(ls='--', lw=0.4)
fS_ax2.set_xlabel('V$_{pR}$ (V)')
fS_ax2.set_ylabel('G$(e^2/h)$')
fS_ax2.set_yscale('linear')
# ax[0].set_xlim(-3.5, -2.4)
fS_ax2.set_ylim(-0.01, 0.12)


# left, bottom, width, height = [1.0, 0.6, 0.2, 0.2]
# inset_ax = fig2.add_axes([left, bottom, width, height])

# inset_ax = inset_axes(f2_ax2,
#                     width="30%", # width = 30% of parent_bbox
#                     height=1., # height : 1 inch
#                     bbox_to_anchor=(0,0,1,1), bbox_transform=f2_ax2.transAxes,
#                     loc=1)
inset_ax = fS_ax4
# mpl.style.use('default')
# labels=['bottom', 'top', 'top']
# ax2=ax.twinx()
offset=0
npoints=[20]
colors = ['b', 'r' ,'k']
for i, d in enumerate([176]):
    dat = load1dFig5(d)
    vplunger = dat[:,1]
    vs = dat[:,2]
    vt = dat[:,4]
    vr = dat[:,6]
    G = 3*vt/(vt+vr)
    alpha = 0.042
    widths = np.zeros((58))
    width_height = np.zeros((58))
    left = np.zeros((58))
    right = np.zeros((58))
    start=2
    peaks, _ = find_peaks(G[start:], prominence=0.08)
    Gpeak = G[start+peaks]
    print('------------')
    print(np.mean(Gpeak))
    print(np.std(Gpeak))
    print('------------')
    Gmax = np.max(G[start:])
    print(np.diff(vplunger[start + peaks]))
#         phase = dat[:, 3]
#         v = 5e-6
#         vqpc = -1.988 - 0.002*i
    widths[:len(peaks)], width_height[:len(peaks)], left[:len(peaks),], right[:len(peaks)] = peak_widths(G[start:], peaks, rel_height=0.5)
#         ax.plot(1*0.03*i+curr[730:930]/v*25813, lw=0.75, label=str(vqpc)+' V', color='k')
    # ax.plot(vplunger[start:], offset*0.005*i+G[start:], 'o-', lw=0.75, markersize=2, label=labels[i])
    
#         ax.plot(1*0.03*i+curr[left[i]]/v*25813, lw=0.75, label=str(vqpc)+' V', color='k')
    # ax.semilogy(vplunger[start+peaks],offset*0.005*i+G[start+peaks],'b*')
    startpeak = 1
    endpeak = 2
    inset_ax.plot(vplunger[start + peaks[startpeak] - 2*npoints[i] : start + peaks[endpeak-1] + 2*npoints[i]], G[start + peaks[startpeak] - 2*npoints[i] : start + peaks[endpeak-1] + 2*npoints[i]], 'bx-', lw=1)#, label='SET')
    
    Gfiterror = np.zeros((16))
    Gfitmean = np.zeros((16))
    weighted_sum = 0
    weight_norm = 0
    for peaknum in range(1,2):
        Vg0 = vplunger[start+ peaks[peaknum]]
#         print(Vg0)
        G0 = np.max(G[start+peaks[peaknum]])
#         alpha = 0.04
        Vg = vplunger[start + peaks[peaknum] - npoints[i] : start + peaks[peaknum] + npoints[i]]
        deltaVg = Vg - Vg0
        Gfit = G[start + peaks[peaknum] - npoints[i] : start + peaks[peaknum] + npoints[i]]
        popt, pcov = curve_fit(fitG, deltaVg, Gfit, p0=[G0, 11000])
        # popt2, pcov2 = curve_fit(fitGauss, deltaVg, Gfit, p0=[G0, 11000/3])
#             popt2, pcov2 = curve_fit(fitGdot, deltaVg, Gfit, p0=[G0, 5000])
        print(alpha*1*(popt[1]**-1)*1e6/86)
        perr = np.sqrt(np.diag(pcov))
        # perr2 = np.sqrt(np.diag(pcov2))
        print(alpha*1*(perr[1]*popt[1]**-2)*1e6/86)
        # print('------------')
        Gfitmean[peaknum] = popt[0]
        Gfiterror[peaknum] = perr[0]
        weighted_sum = weighted_sum + Gfitmean[peaknum]/(Gfiterror[peaknum]**2)
        weight_norm = weight_norm + 1/(Gfiterror[peaknum]**2)
        # print('------------')
#         print(alpha*0.5*(popt2[1]**-1)*1e6/86)

#         print(popt[1])
        if peaknum==startpeak:
            inset_ax.plot(Vg, 0*0.005*i+fitG(deltaVg, popt[0], popt[1]), 'r--', lw=2, label=f'{1e3*alpha*1*(popt[1]**-1)*1e6/86:.2f} $\pm$ {1e3*alpha*1*(perr[1]*popt[1]**-2)*1e6/86:.2f} mK')
            # inset_ax.plot(Vg, 0*0.005*i+fitGauss(deltaVg, popt2[0], popt2[1]), 'k--', lw=2, label=f'{1e3*alpha*1*(popt2[1]**-1)*1e6/86:.2f} $\pm$ {1e3*alpha*1*(perr2[1]*popt2[1]**-2)*1e6/86:.2f} mK')
        else:
            inset_ax.plot(Vg, 0*0.005*i+fitG(deltaVg, popt[0], popt[1]), 'r--', lw=2)#, label='SET')

    # ax.plot(vplunger[:]+ 0.0*(i//2),0.0*i+ G[:], 'b*-')
    inset_ax.legend()
    inset_ax.legend(loc='lower right', framealpha=1)
print(f'Weighted average of peak heights = {weighted_sum/weight_norm:.3}')
print(f'Error in estimation of peak heights = {np.sqrt(1/weight_norm):.3}')
inset_ax.grid(ls='--', lw=0.4)
inset_ax.set_xlabel('V$_{pR}$ (V)')
inset_ax.set_ylabel('G$(e^2/h)$')
inset_ax.set_yscale('log')
# ax[0].set_xlim(-3.5, -2.4)
inset_ax.set_ylim(1e-3, 0.21)

fS_ax3.text(-0.1, 1, "(c)", fontsize=14, va="bottom", ha="right", transform=fS_ax3.transAxes)
fS_ax4.text(-0.1, 1, "(d)", fontsize=14, va="bottom", ha="right", transform=fS_ax4.transAxes)
################################
dat = load2dFig5(262, 81)
plunger = dat['1']
idc = dat['2']
vr = dat['3']
vt = dat['5']
vtdc = dat['9']
vsdc = dat['10']
dcoffset = np.mean(vsdc[40,:])
G = 3*vt/(vt+vr)

im = fS_ax3.pcolormesh(plunger, (vsdc - 5e-6)*1e6, np.abs(G),vmin=1e-3, vmax=0.15, cmap='magma', rasterized=True)


fS_ax3.set_xlabel('$V_{pR}$ (V)')
fS_ax3.set_ylabel('$V_{dc}$ ($\mu$V)')
fS_ax3.set_ylim(-170, 170)
cax = fS_ax3.inset_axes([1.01, 0, 0.02, 1])
cbar = figS.colorbar(im, cax=cax, extend='max')
cbar.ax.set_ylabel('$G (e^2/h)$')
####################################################




skunk.connect(fS_ax1, 'sk2')

# svg = skunk.pltsvg(fig=fig2)
svg = skunk.insert(
    {  
        'sk2': 'SupplementSchematicImport.svg'
            
    })

# fig2.set_constrained_layout(False)
# fig2.tight_layout()

# fig1.canvas.draw()
# # we want the legend included in the bbox_inches='tight' calcs.
# # cbar1.set_in_layout(True)
# # cbar2.set_in_layout(True)
# # we don't want the layout to change at this point.
#fig1.tight_layout()
#with open('/Users/praveen/Downloads/FIG2A.svg', 'w') as f:
 #   f.write(svg)
    
skunk.display(svg)
# cairosvg.svg2pdf(bytestring=svg, write_to='/Users/praveen/Library/CloudStorage/GoogleDrive-prvn@stanford.edu/Shared drives/GGG GDrive/QDots-2/Papers/Hybrid Dot APL/figureS.pdf')
# fig1.savefig('Figure1.pdf', dpi=300)
# ax[0,2].axis('off')
# ax[1,2].axis('off')
# plt.tight_layout()

# plt.savefig("Figure4.eps")